# 01 · Pipeline de extracción: FASTA + CSV → MongoDB

**Entregable 2** · Bases de Datos II · PanTEon Database v1.6.1

Rama: `feature/pipeline-limpieza`

---

## Alcance (guía, §4.2)

- [ ] Leer y validar el FASTA: separar `ID`, `CLASE/ORDEN/SUPERFAMILIA` y `@Especie`; verificar caracteres válidos, presencia de `N` y longitudes.
- [ ] Leer y limpiar el CSV: conservar **solo** las 9 columnas (`Kingdom`, `Phylum`, `Class`, `Order`, `Family`, `Species`, `Source`, `Source 2`, `Notes`) y descartar `Library?`, `Selected`, `TE_count`, `MCH_Parameters`.
- [ ] Normalizar los nombres de especie en ambos lados y cruzar por `Species`.
- [ ] Reportar huérfanos en **ambos** sentidos (FASTA sin CSV y CSV sin FASTA).
- [ ] Calcular al menos dos atributos derivados por secuencia: longitud, %GC, %N.
- [ ] Calcular los *k*-mers de `k=1` a `k=6` y **guardarlos en MongoDB**, definiendo y documentando la política para ventanas con `N` u otros caracteres fuera de `{A, C, G, T}`.
- [ ] Cargar con `insert_many` o `bulk_write` (`pymongo`).
- [ ] Dejar el notebook ejecutable de principio a fin (requisito de reproducibilidad).

## Reglas de trabajo

1. Validar el diseño con un subconjunto de ~5 especies antes de escalar a las ~240.000 secuencias.
2. No subir datos crudos: `data_raw/` está en `.gitignore`.
3. Cada decisión de diseño se anota en `docs/decisiones.md` **en el momento**, no al final.


In [81]:
import sys, os
print(sys.executable)
print(os.getcwd())
print(os.listdir("../data_raw"))

C:\Users\camil\.conda\envs\transponibles\python.exe
C:\Users\camil\Documents\BDProyectoCorteII\panteon-mongodb-bd2\notebooks
['PanTEon_Database_metadata_v1.6.1.csv', 'PanTEon_Database_v1.6.1.fasta']


In [82]:
import pandas as pd

df = pd.read_csv("../data_raw/PanTEon_Database_metadata_v1.6.1.csv")
df_prueba = df.sample(9, random_state=1)
df_prueba

,Kingdom,Phylum,Class,Order,Family,Species,Library?,Source,Source 2,Notes,Selected,TE_count,MCH_Parameters
231,Fungi,Basidiomycota,Agaricomycetes,Agaricales,Physalacriaceae,Armillaria cepistipes,Complete,RM2+MCH,NaN,"ENSEMBL ""2023""",RM2+MCH,28,"x=10, e=500, c=1"
1443,Fungi,Basidiomycota,Agaricomycetes,Russulales,Russulaceae,Lactarius hatsudake,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,11,"x=0, e=500, c=1"
1699,Fungi,Mucoromycota,Mortierellomycetes,Mortierellales,Mortierellaceae,Mortierella hygrophila,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,5,"x=0, e=500, c=1"
685,Animalia,Chordata,Mammalia,Cingulata,Chlamyphoridae,Chaetophractus vellerosus,Complete,Dfam,NaN,Curated Dfam,Dfam,65,NaN
2209,Fungi,Basidiomycota,Agaricomycetes,Gomphales,Gomphaceae,Ramaria rubella,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,28,"x=0, e=500, c=1"
1551,Animalia,Chordata,Mammalia,Primates,Cercopithecidae,Macaca fascicularis,Complete,Dfam,NaN,Release 1.2,Dfam,56,NaN
2572,Fungi,Ascomycota,Sordariomycetes,Hypocreales,Ophiocordycipitaceae,Tolypocladium inflatum,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,1,"x=0, e=500, c=1"
1711,Fungi,Mucoromycota,Mucoromycetes,Mucorales,Mucoraceae,Mucor racemosus,Complete,RM2+MCH,NaN,"ENSEMBL ""2023""",RM2+MCH,24,"x=10, e=500, c=1"
2699,Fungi,Ascomycota,Dothideomycetes,Venturiales,Venturiaceae,Venturia nashicola,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,5,"x=0, e=500, c=1"


In [83]:
print(df.shape)# Devuelve una pareja de filas y columnas, sirve para saber el tamaño real de lo que se tiene
print(df.columns.tolist())#(df.columns)son los nombres de las columnas (.tolist()) los convierte en una lista de python, para verlo en una sola linea 


(2790, 13)
['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Species', 'Library?', 'Source', 'Source 2', 'Notes', 'Selected', 'TE_count', 'MCH_Parameters']


In [84]:
#crea una tabla donde si esta vacio es true y los suma y si es no esta vacio o sea false 0
df.isna().sum()

Kingdom              0
Phylum               3
Class               49
Order               29
Family              76
Species              0
Library?             0
Source               0
Source 2          2740
Notes                6
Selected             0
TE_count             0
MCH_Parameters     386
dtype: int64

In [85]:
# df["Kingdom"] selecciona esa columna.
#.value_counts() cuenta cuántas veces aparece cada valor distinto.
#dropna=False hace que también cuente los vacíos. Sin eso, los NaN se ocultarían y no los verías.
df["Kingdom"].value_counts(dropna=False)

Kingdom
Fungi       1559
Animalia     780
Plantae      451
Name: count, dtype: int64

In [86]:
#.str.strip() quita los espacios al inicio y al final de cada nombre.
#!= compara el nombre original con el recortado: da True donde son distintos, o sea donde había espacios de más.
#.sum() cuenta los True.
(df["Species"] != df["Species"].str.strip()).sum()

np.int64(0)

In [87]:
#Compara cada nombre con su versión en minúsculas. Como todos los nombres de especie empiezan con mayúscula (Macaca fascicularis), todos difieren.
(df["Species"] != df["Species"].str.lower()).sum()

np.int64(2790)

In [88]:
#.str.lower() pasa todo a minúsculas.
#.duplicated() marca cada nombre que ya apareció antes.
#.sum() los cuenta.
df["Species"].str.lower().duplicated().sum()

np.int64(0)

In [89]:
#Cuenta cuántas especies hay por origen, incluyendo vacíos.
df["Source"].value_counts(dropna=False)

Source
RM2+MCH    2404
Dfam        386
Name: count, dtype: int64

In [90]:
df[["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]]
df_csv_9col = df[["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]].copy()

In [91]:
print(df_csv_9col.shape)
df_csv_9col.isna().sum()

(2790, 9)


Kingdom        0
Phylum         3
Class         49
Order         29
Family        76
Species        0
Source         0
Source 2    2740
Notes          6
dtype: int64

In [92]:
# Se normaliza Species con strip(): si el CSV y el FASTA difieren por un espacio sobrante, el cruce no los une
df_csv_9col["Species"] = df_csv_9col["Species"].str.strip() #Se Sobreescribe la columna con su version recortada

In [93]:
(df_csv_9col["Species"] != df_csv_9col["Species"].str.strip()).sum()

np.int64(0)

In [94]:
#EXPLORACIÓN DE VACÍOS Y PROCEDENCIA

In [95]:
#Vacios
df_csv_9col[df_csv_9col["Phylum"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
1310,Fungi,NaN,Agaricomycetes,Agaricales,Hymenogastraceae,Hebeloma cylindrosporum,RM2+MCH,NaN,B-GUT
1617,Animalia,NaN,NaN,NaN,NaN,Metazoa,Dfam,Repbase,No especie Repbase
2143,Animalia,NaN,NaN,NaN,NaN,Protostomia,Dfam,NaN,Dfam no especie


In [96]:
df_csv_9col[df_csv_9col["Class"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
56,Fungi,Oomycota,NaN,Albuginales,Albuginaceae,Albugo candida,RM2+MCH,NaN,B-GUT
57,Fungi,Oomycota,NaN,Albuginales,Albuginaceae,Albugo laibachii,RM2+MCH,NaN,B-GUT
106,Plantae,Angiosperma,NaN,Amborellales,Amborellaceae,Amborella trichopoda,RM2+MCH,Repbase,Uncurated Dfam
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
170,Fungi,Oomycota,NaN,Saprolegniales,Saprolegniaceae,Aphanomyces astaci,RM2+MCH,NaN,B-GUT
204,Plantae,Angiosperma,NaN,Ranunculales,Ranunculaceae,Aquilegia eximia,RM2+MCH,NaN,Uncurated Dfam
484,Plantae,Angiosperma,NaN,Caryophyllales,Amaranthaceae,Beta vulgaris,RM2+MCH,Repbase,"ENSEMBL ""2023"""
520,Plantae,Angiosperma,NaN,Poales,Poaceae,Bonia amplexicaulis,RM2+MCH,NaN,Uncurated Dfam
547,Plantae,Angiosperma,NaN,Poales,Poaceae,Brachypodium distachyon,RM2+MCH,Repbase,Uncurated Dfam
583,Fungi,Mucoromycota,NaN,Calcarisporiellales,Calcarisporiellaceae,Calcarisporiella thermophila,RM2+MCH,NaN,B-GUT


In [97]:
df_csv_9col[df_csv_9col["Order"].isna()]

,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
509,Fungi,Chytridiomycota,Chytridiomycetes,NaN,NaN,Blyttiomyces helicus,RM2+MCH,NaN,B-GUT
830,Fungi,Ascomycota,Dothideomycetes,NaN,NaN,Coniosporium apollinis,RM2+MCH,NaN,B-GUT
873,Fungi,Ascomycota,Dothideomycetes,NaN,NaN,Cryomyces minteri,RM2+MCH,NaN,B-GUT
975,Fungi,Ascomycota,Dothideomycetes,NaN,Zopfiaceae,Didymocrea sadasivanii,RM2+MCH,NaN,B-GUT
1031,Fungi,Microsporidia,NaN,NaN,Culicosporidae,Edhazardia aedis,RM2+MCH,NaN,B-GUT
1129,Animalia,Chordata,NaN,NaN,NaN,Euteleostomi,Dfam,Repbase,No especie Repbase
1300,Fungi,Microsporidia,Dubosqiidae,NaN,NaN,Hamiltosporidium magnivora,RM2+MCH,NaN,B-GUT
1301,Fungi,Microsporidia,Dubosqiidae,NaN,NaN,Hamiltosporidium tvaerminnensis,RM2+MCH,NaN,B-GUT
1330,Fungi,Microsporidia,NaN,NaN,Hepatosporidae,Hepatospora eriocheir,RM2+MCH,NaN,B-GUT


In [98]:
df_csv_9col[df_csv_9col["Family"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
10,Fungi,Ascomycota,Sordariomycetes,Amplistromataceae,NaN,Acidothrix acidophila,RM2+MCH,NaN,B-GUT
39,Animalia,Chordata,Mammalia,No trobat,NaN,Afrotheria,Dfam,NaN,Dfam no especie
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
118,Fungi,Ascomycota,Leotiomycetes,Helotiales,NaN,Amylocarpus encephaloides,RM2+MCH,NaN,B-GUT
164,Fungi,Ascomycota,Sordariomycetes,Xylariales,NaN,Anthostomelloides leucospermi,RM2+MCH,NaN,B-GUT
...,...,...,...,...,...,...,...,...,...
2577,Fungi,Microsporidia,Pleistophoridae,NaN,NaN,Trachipleistophora hominis,RM2+MCH,NaN,B-GUT
2677,Fungi,Basidiomycota,Ustilaginomycetes,Ustilaginales,NaN,Ustilago maydis,RM2+MCH,NaN,B-GUT
2704,Fungi,Ascomycota,Dothideomycetes,Pleosporales,NaN,Verruculina enalia,RM2+MCH,NaN,B-GUT
2739,Animalia,Chordata,Mammalia,No trobat,NaN,Xenarthra,Dfam,NaN,Dfam no especie


In [99]:
df_csv_9col["Notes"].value_counts(dropna=False)

Notes
B-GUT                 1479
ENSEMBL "2023"         375
ENSEMBL "2025"         373
Curated Dfam           293
Uncurated Dfam         165
Release 1.2             43
Dfam no especie         34
APTEdb                  12
No especie Repbase      10
NaN                      6
Name: count, dtype: int64

In [100]:
df_csv_9col["Order"].value_counts(dropna=False).head(30)

Order
Eurotiales           267
Agaricales           127
Hypocreales          101
Pleosporales          78
Pezizales             73
Polyporales           66
Rodentia              64
Boletales             62
Sordariales           61
Poales                60
Mucorales             57
Helotiales            56
Primates              54
Artiodactyla          47
Glomerellales         46
Passeriformes         42
Russulales            38
Xylariales            36
Chiroptera            36
Myrtales              33
Carnivora             32
Lepidoptera           31
NaN                   29
Saccharomycetales     28
Malpighiales          26
Onygenales            24
Mortierellales        23
Fabales               23
Rosales               23
Lamiales              22
Name: count, dtype: int64

In [101]:
df_csv_9col[df_csv_9col["Class"].isna()]["Kingdom"].value_counts()

Kingdom
Fungi       21
Plantae     15
Animalia    13
Name: count, dtype: int64

In [102]:
df_csv_9col["Order"].value_counts(dropna=False).tail(15)

Order
Thysanoptera                               1
Hypnales                                   1
Scombriformes                              1
Harpacticoida                              1
Trichinellida                              1
Phaeotrichales                             1
Celastrales                                1
Trogoniformes                              1
Ulvales                                    1
Umbelopsidales                             1
Hirudinida                                 1
Teloschistales                             1
Pandanales                                 1
Xylonales                                  1
No order Dothideomycetes incertae sedis    1
Name: count, dtype: int64

In [103]:
marcadores = ["No trobat", "No order"]
df_csv_9col.isin(marcadores).sum()

Kingdom      0
Phylum       0
Class        0
Order       12
Family       0
Species      0
Source       0
Source 2     0
Notes        0
dtype: int64

In [104]:
df_csv_9col[df_csv_9col["Order"].str.startswith("No ", na=False)]["Order"].value_counts()

Order
No trobat                                       7
No order                                        5
No ordre NCBI                                   2
No Orden No trobat                              2
No ordre NCBI Caenogastropoda incertae sedis    1
No order (eupecaria incertae sedis)             1
No order Dothideomycetes incertae sedis         1
Name: count, dtype: int64

In [105]:
df_csv_9col["Order"].isna().sum()

np.int64(29)

In [108]:
import numpy as np
df_csv_9col.loc[df_csv_9col["Order"].str.startswith("No ", na=False) , "Order" ] = np.nan

In [109]:
print(df_csv_9col["Order"].isna().sum())
print(df_csv_9col.shape)

48
(2790, 9)


In [110]:
df_csv_9col[df_csv_9col["Class"].str.startswith("No ", na=False)]["Class"].value_counts()

Series([], Name: count, dtype: int64)

In [111]:
df_csv_9col[df_csv_9col["Family"].str.startswith("Unknown ", na=False)]["Family"].value_counts()

Series([], Name: count, dtype: int64)

In [112]:
#ARCHIVO FASTA
from Bio import SeqIO
ruta_fasta = "../data_raw/PanTEon_Database_v1.6.1.fasta"


In [113]:
for i, registro in enumerate( SeqIO.parse(ruta_fasta, "fasta")):
    #registro.id: lo que hay en el encabezado después del > y hasta el primer espacio.
    print(registro.id)
    #registro.description: el encabezado completo, sin el >.
    print(registro.description)
    #registro.seq: la secuencia de nucleótidos. Es un objeto Seq, que se comporta parecido a un texto.
    print(len(registro.seq))
    if i == 4:
        break

PDB00000001#CLASSII/HELITRON/HELITRON
PDB00000001#CLASSII/HELITRON/HELITRON @Halicephalobus mephisto
1991
PDB00000002#CLASSII/HELITRON/HELITRON
PDB00000002#CLASSII/HELITRON/HELITRON @Halicephalobus mephisto
1689
PumCon-1.11#CLASSI/SINE/TRNA
PumCon-1.11#CLASSI/SINE/TRNA @Puma concolor
235
PumCon-1.141#CLASSII/TIR/TC1MARINER
PumCon-1.141#CLASSII/TIR/TC1MARINER @Puma concolor
340
PumCon-1.14#CLASSI/SINE/TRNA
PumCon-1.14#CLASSI/SINE/TRNA @Puma concolor
233


In [114]:
#Separacion de la cadena del ID de la clasificacion, la clasificacion de cada especie y la clase, orden y superfamilia entre si
def separar_cadena(cadena):
    #split("@", 1): corta en el primer @ y da dos pedazos (izquierda y especie). El 1 limita a un solo corte.
    izquierda, especie = cadena.split("@", 1)
    #split("#", 1): igual, separa ID y clasificación.
    id_seq, clasificacion = izquierda.split("#", 1)
    #split("/"): separa en tres niveles. Si algún encabezado no tiene exactamente tres, da error; eso lo manejamos en el paso 3.
    clase_fasta, orden_fasta, superfamilia_fasta = clasificacion.strip().split("/")
    #strip() quita espacios sobrantes en cada pedazo.
    return (id_seq.strip(), clase_fasta.strip(), orden_fasta.strip(), superfamilia_fasta.strip(), especie.strip())

In [115]:
#Recorrer el FASTA para saber cuántos encabezados no cumplen el patrón, Es el paso que que alimenta el reporte de inconsistencias.
total = 0
malos = []
for registro in SeqIO.parse("../data_raw/PanTEon_Database_v1.6.1.fasta", "fasta"):
    total += 1
    try:
        separar_cadena(registro.description)
    except ValueError:
        (malos.append(registro.description))
print(total)
print(len(malos))
print(malos[:10])
        

239946
0
[]


In [116]:
#Construir tabla de secuencias , transformar y guardar
filas = []
for registro in SeqIO.parse("../data_raw/PanTEon_Database_v1.6.1.fasta", "fasta"):
    #las 5 partes que devuelve tu funcion
    id_seq, clase_fasta, orden_fasta, superfamilia_fasta, especie = separar_cadena(registro.description)
    #Secuencia como texto en mayusculas
    secuencia = str(registro.seq).upper()
    longitud = len(secuencia)
    n_count =  secuencia.count("N")
    gc_count = secuencia.count("G") + secuencia.count("C")

    filas.append((id_seq, clase_fasta, orden_fasta, superfamilia_fasta, especie, longitud, n_count, gc_count))
   # if len(filas) == 1000:
   #    break
df_secuencias = pd.DataFrame(filas, columns = ["id_seq", "clase_fasta", "orden_fasta", "superfamilia_fasta", "especie", "longitud", "n_count", "gc_count"])
print(df_secuencias.shape)
df_secuencias.head()

(239946, 8)


,id_seq,clase_fasta,orden_fasta,superfamilia_fasta,especie,longitud,n_count,gc_count
0,PDB00000001,CLASSII,HELITRON,HELITRON,Halicephalobus mephisto,1991,0,628
1,PDB00000002,CLASSII,HELITRON,HELITRON,Halicephalobus mephisto,1689,0,533
2,PumCon-1.11,CLASSI,SINE,TRNA,Puma concolor,235,0,120
3,PumCon-1.141,CLASSII,TIR,TC1MARINER,Puma concolor,340,0,135
4,PumCon-1.14,CLASSI,SINE,TRNA,Puma concolor,233,0,123


In [117]:
df_secuencias["id_seq"].duplicated().sum()

np.int64(0)

In [118]:
df_secuencias["n_count"].gt(0).sum()

np.int64(54837)

In [119]:
(df_secuencias["gc_count"] > df_secuencias["longitud"]).sum()

np.int64(0)

In [120]:
df_secuencias["clase_fasta"].value_counts(dropna=False)

clase_fasta
CLASSI     194774
CLASSII     45172
Name: count, dtype: int64

In [121]:
df_secuencias["orden_fasta"].value_counts(dropna=False)

orden_fasta
LTR           119368
LINE           68382
TIR            36471
HELITRON        5075
SINE            3634
MITE            3456
DIRS            2933
PLE              344
RETROPOSON       113
CRYPTON          112
MAVERICK          58
Name: count, dtype: int64

In [122]:
#NORMALIZAR especie en el FASTA, se normaliza para que el cruce compare nombres iguales y no nombres que solo se parecen
df_secuencias["especie"] = df_secuencias["especie"].str.strip() #Se Sobreescribe la columna con su version recortada

In [123]:
(df_secuencias["especie"] != df_secuencias["especie"].str.strip()).sum()

np.int64(0)

In [124]:
#Sacar los 2 conjuntos de especies
df_secuencias["especie"].unique()

<StringArray>
[      'Halicephalobus mephisto',                 'Puma concolor',
             'Takifugu rubripes',               'Myotis brandtii',
        'Galeopterus variegatus',            'Spilogale gracilis',
         'Certhia brachydactyla',           'Myripristis murdjan',
           'Anabarilius grahami',          'Amphiduros pacificus',
 ...
     'Scheffersomyces insectosa',          'Aspergillus bombycis',
      'Scheffersomyces shehatae',  'Heterodoassansia hygrophilae',
        'Zygotorulaspora mrakii', 'Ustanciosporium gigantosporum',
       'Penicillium chermesinum',         'Talaromyces rugulosus',
      'Lepidopterella palustris',         'Aspergillus parafelis']
Length: 2790, dtype: str

In [137]:
df_secuencias["especie"] = df_secuencias["especie"].str.replace(".", "", regex=False)
df_csv_9col["Species"]  = df_csv_9col["Species"].str.replace(".", "", regex=False)

In [138]:
esp_fasta = set(df_secuencias["especie"].unique())

In [139]:
esp_csv = set(df_csv_9col["Species"].unique())

In [140]:
print(len(esp_fasta))
print(len(esp_csv))

2790
2790


In [141]:
#LAS 3 LISTAS
en_ambos_existe = esp_fasta & esp_csv
solo_fasta = esp_fasta - esp_csv
solo_csv = esp_csv - esp_fasta 

print(len(en_ambos_existe))
print(len(solo_fasta))
print(len(solo_csv))

2790
0
0


In [143]:
print(sorted(en_ambos_existe))
print(sorted(solo_fasta))
print(sorted(solo_csv))

['Abortiporus biennis', 'Abrus pulchellus subsp cantoniensis', 'Absidia caerulea', 'Absidia glauca', 'Absidia repens', 'Acanthaster planci', 'Acanthoscelides obtectus', 'Acephala macrosclerotiorum', 'Acer negundo', 'Achaetomium macrosporum', 'Achlya flavicornis', 'Acidothrix acidophila', 'Acinonyx jubatus', 'Acipenser ruthenus', 'Acomys cahirinus', 'Acridotheres javanicus', 'Acridotheres tristis', 'Acrocalymma vagum', 'Acrocera orbiculus', 'Acromyrmex echinatior', 'Acropora acuminata', 'Acropora millepora', 'Acropora nasuta', 'Acrossocheilus wenchowensis', 'Actinia tenebrosa', 'Actinidia chinensis', 'Actinidia deliciosa', 'Actinidia eriantha', 'Actinomortierella ambigua', 'Actinomortierella capitata', 'Actinomortierella wolfii', 'Adineta ricciae', 'Aegilops bicornis', 'Aegilops longissima', 'Aegilops searsii', 'Aegilops speltoides', 'Aegilops tauschii', 'Aegithalos caudatus', 'Aegotheles bennettii', 'Aethina tumida', 'Afrotheria', 'Agaricus bisporus', 'Agelaius phoeniceus', 'Agelas oro

In [144]:
print("Coccomyxa sp obi" in en_ambos_existe)

True


In [148]:
#Convertir los conteos en porcentajes , df_secuencias guarda por cada secuencias conteos
df_secuencias["pct_n"] = df_secuencias["n_count"] / df_secuencias["longitud"] * 100

In [150]:
df_secuencias["pct_gc"] = df_secuencias["gc_count"] / df_secuencias["longitud"] * 100

In [153]:
df_secuencias[["pct_gc", "pct_n"]].describe()

,pct_gc,pct_n
count,239946.000000,239946.000000
mean,43.016822,0.379045
std,7.536644,1.186139
min,11.032532,0.000000
25%,38.025603,0.000000
50%,42.147651,0.000000
75%,47.496452,0.000000
max,78.902954,50.062735


In [152]:
df_secuencias.head()

,id_seq,clase_fasta,orden_fasta,superfamilia_fasta,especie,longitud,n_count,gc_count,pct_n,pct_gc
0,PDB00000001,CLASSII,HELITRON,HELITRON,Halicephalobus mephisto,1991,0,628,0.0,31.541939
1,PDB00000002,CLASSII,HELITRON,HELITRON,Halicephalobus mephisto,1689,0,533,0.0,31.557134
2,PumCon-1.11,CLASSI,SINE,TRNA,Puma concolor,235,0,120,0.0,51.063830
3,PumCon-1.141,CLASSII,TIR,TC1MARINER,Puma concolor,340,0,135,0.0,39.705882
4,PumCon-1.14,CLASSI,SINE,TRNA,Puma concolor,233,0,123,0.0,52.789700


In [154]:
df_secuencias["superfamilia_fasta"].value_counts(dropna=False).head(30)

superfamilia_fasta
COPIA           50849
GYPSY           43963
L1              23821
ERV             12202
CR1             11234
L2              10962
TC1MARINER      10729
HAT             10648
TIR              5983
RTE              5143
HELITRON         5075
LARD             4588
TAD1             3854
MITE             3456
PIFHARBINGER     3010
CAULIMOVIRUS     2962
CMC              2352
MULE             2306
DIRS             2067
LINE             2066
BELPAO           1995
I                1990
REX-BABAR        1923
JOCKEY           1605
LTR              1499
LOA              1368
RTEX             1314
TRIM             1310
TRNA             1271
R1               1176
Name: count, dtype: int64

In [155]:
#: busca valores que no parezcan una superfamilia real
# value_counts() cuenta cuántas secuencias hay por cada superfamilia.
# dropna=False hace que también cuente los vacíos (NaN), si los hubiera.
# tail(30) muestra las 30 con MENOS secuencias (el final de la lista ordenada de mayor a menor).
# Sirve para detectar variantes raras, errores de escritura o valores vacíos
# que no se ven en el top 30.
df_secuencias["superfamilia_fasta"].value_counts(dropna=False).tail(30)

superfamilia_fasta
DONG-R4       440
MIR           346
PLE           344
P             341
ACADEM-1      173
R2            161
HERO          161
KOLOBOK       155
MERLIN        142
ID            126
RETROPOSON    113
CRYPTON       112
B4            107
5S             91
B2             68
MAVERICK       58
DUALEN         56
tRNA           56
ZISUPTON       54
SOLA           54
ZATOR          23
PROTO1         19
IS3EU          15
DECEIVER       13
P-FUNGI        11
DADA           10
U               8
GINGER          6
PIF             5
ERV1            4
Name: count, dtype: int64

In [156]:
# nunique() cuenta cuántos valores DISTINTOS tiene la columna.
# Sirve para saber el tamaño de la jerarquía: cuántas superfamilias existen en total.
df_secuencias["superfamilia_fasta"].nunique()

66

In [157]:
# Compara, fila por fila, la superfamilia con el orden: da True donde son iguales
# (por ejemplo CLASSII/HELITRON/HELITRON) y False donde son distintos.
# sum() cuenta los True (Python cuenta True como 1 y False como 0).
# Sirve para saber en cuántas secuencias NO se asignó una superfamilia más específica
# que el orden. Es un dato útil para el reporte de calidad.
(df_secuencias["superfamilia_fasta"] == df_secuencias["orden_fasta"]).sum()

np.int64(21544)

In [158]:
# str.strip() quita espacios al inicio y al final de cada texto.
# Después comparamos con "" (texto vacío): da True donde la superfamilia
# estaba vacía o tenía solo espacios.
# sum() cuenta cuántos casos hay. Debe dar 0 si no hay campos vacíos disfrazados.
(df_secuencias["superfamilia_fasta"].str.strip() == "").sum()

np.int64(0)

In [159]:
df_secuencias["superfamilia_fasta"].str.upper().nunique()

65

In [160]:
# Agrupa por la versión en mayúsculas y muestra las escrituras originales de cada grupo.
# Solo se ven los grupos con más de una escritura distinta.
g = df_secuencias.groupby(df_secuencias["superfamilia_fasta"].str.upper())["superfamilia_fasta"].unique()
g[g.apply(len) > 1]

superfamilia_fasta
TRNA    [TRNA, tRNA]
Name: superfamilia_fasta, dtype: object

In [161]:
df_secuencias[df_secuencias["superfamilia_fasta"].str.upper() == "TRNA"].groupby(
    ["superfamilia_fasta", "clase_fasta", "orden_fasta"]).size()

superfamilia_fasta  clase_fasta  orden_fasta
TRNA                CLASSI       SINE           1271
tRNA                CLASSI       SINE             56
dtype: int64

In [164]:
df_secuencias["superfamilia_fasta"] = df_secuencias["superfamilia_fasta"].replace({"tRNA":"TRNA"})

In [165]:
df_secuencias["superfamilia_fasta"].nunique()

65

In [166]:
(df_secuencias["superfamilia_fasta"] == "tRNA").sum()

np.int64(0)

In [167]:
(df_secuencias["superfamilia_fasta"] == "TRNA").sum()

np.int64(1327)

In [168]:
# Objetivo: detectar superfamilias que se repiten bajo órdenes distintos.
# Si una superfamilia está en varios órdenes, agrupar solo por superfamilia mezclaría grupos.

# groupby agrupa las filas por superfamilia.
# nunique() cuenta cuántos órdenes distintos tiene cada una.
# sort_values(ascending=False) deja arriba las que tienen más órdenes.
# head(10) muestra solo las 10 primeras.
df_secuencias.groupby("superfamilia_fasta")["orden_fasta"].nunique().sort_values(ascending=False).head(10)

superfamilia_fasta
5S              1
ACADEM-1        1
ALU             1
B2              1
B4              1
BELPAO          1
CAULIMOVIRUS    1
CMC             1
COPIA           1
CR1             1
Name: orden_fasta, dtype: int64

In [169]:
# Objetivo: marcar (sin borrar) qué filas del CSV son especies reales.
# True = especie real; False = grupo taxonómico que no es especie (Metazoa, Tetrapoda...).
# Se conservan todas porque el FASTA tiene secuencias de estos grupos.

# isin([...]) da True donde Notes coincide con alguno de los dos textos "no especie".
# ~ invierte el resultado: así las 44 filas quedan en False y el resto en True.
df_csv_9col["es_especie"] = ~df_csv_9col["Notes"].isin([ "Dfam no especie" , "No especie Repbase" ])

In [170]:
# Cuenta los True: debe dar 2746 (2790 - 44).
df_csv_9col["es_especie"].sum()

np.int64(2746)

In [171]:
# Cuenta los False: debe dar 44 (34 + 10).
(~df_csv_9col["es_especie"]).sum()

np.int64(44)

In [172]:
# Confirma que no se perdieron filas; ahora son 10 columnas (las 9 + la marca).
df_csv_9col.shape

(2790, 10)

In [174]:
# Muestra algunas filas marcadas False para comprobar que son grupos y no especies.
df_csv_9col[~df_csv_9col["es_especie"]][["Species", "Notes"]]

,Species,Notes
39,Afrotheria,Dfam no especie
110,Amniota,Dfam no especie
243,Artiodactyla,Dfam no especie
521,Boreoeutheria,Dfam no especie
630,Carnivora,Dfam no especie
641,Catarrhini,No especie Repbase
705,Chiroptera,Dfam no especie
725,Chrysemys,Dfam no especie
929,Danio,Dfam no especie
931,Danionidae,Dfam no especie
